### Import dependencies

In [ ]:
pip install opencv-python pandas numpy tqdm

In [21]:
from pathlib import Path
from datetime import datetime
import re
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import random

/Users/alopias/Desktop/Huyen-deePi/deePi-laptop/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Establish paths and some settings

In [96]:
ROOT = Path(r"/Users/alopias/Desktop/deePi-video-processing/converted-videos") 
OUT = Path(r"/Users/alopias/Desktop/Huyen-deePi/deePi-laptop")

BLACK_FRAME_DIR = OUT / "black_frames"
OUT.mkdir(parents=True, exist_ok=True)
BLACK_FRAME_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Settings
# -------------------------
TARGET_BLACK_FRAMES = 150
FPS = 26
MIN_SECONDS_APART = 10
MAX_FRAMES_PER_VID = 6
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
VIDEO_RE = re.compile(r"^\d{12}\.mp4$")

Black frame threshold settings 

In [ ]:
BLACK_RULE = {
    "mean_brightness_max": 10.0,
    "median_brightness_max": 8.0,
    # Most pixels should still be dark
    "p99_brightness_max": 28.0,
    "p999_brightness_max": 60.0,
    # Allow a tiny amount of bright pixels/noise
    "pct_pixels_above_20_max": 0.015,    # <= 1.5%
    "pct_pixels_above_40_max": 0.003,    # <= 0.3%
    "pct_pixels_above_60_max": 0.0008,   # <= 0.08%
    # Allow some texture/noise
    "entropy_max": 4.0,
    # Reject larger bright blobs, but do not reject tiny particles/noise
    "largest_bright_component_area_max": 3000,
}

### Get metadata, construct frame index table, compute frame metrics, etc. (helper functions)

In [98]:
def parse_video_datetime(video_path):
    """
    Parses filename like 202306222131.mp4
    Into: year=2023, month=6, day=22, hour=21, minute=31
    """
    dt = datetime.strptime(video_path.stem, "%Y%m%d%H%M")
    return {"year": dt.year, "month": dt.month, "day": dt.day, "hour": dt.hour, "minute": dt.minute, "datetime": dt}

In [99]:
def build_video_table(root):
    """
    Builds an internal table of videos.
    This is only used inside Python.
    It is NOT saved as an output CSV.
    """
    rows = []
    for camera_folder in sorted(root.iterdir()):
        if not camera_folder.is_dir():
            continue
        camera = camera_folder.name
        for video_path in sorted(camera_folder.glob("*.mp4")):
            if not VIDEO_RE.match(video_path.name):
                continue
            dt_info = parse_video_datetime(video_path)
            rows.append({
                "camera": camera,
                "video": video_path.name,
                "video_path": video_path,
                **dt_info})
    return pd.DataFrame(rows)


In [ ]:
def compute_frame_metrics(frame_bgr):
    """
    Computes brightness metrics for one frame. Includes whole-frame darkness metrics plus localized bright-object metrics.
    """
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

    mean_brightness = float(gray.mean())
    median_brightness = float(np.median(gray))
    max_brightness = int(gray.max())

    p95_brightness = float(np.percentile(gray, 95))
    p99_brightness = float(np.percentile(gray, 99))
    p999_brightness = float(np.percentile(gray, 99.9))

    pct_pixels_above_20 = float((gray > 20).mean())
    pct_pixels_above_40 = float((gray > 40).mean())
    pct_pixels_above_60 = float((gray > 60).mean())
    pct_pixels_above_80 = float((gray > 80).mean())

    # Entropy: flat black images have low entropy.
    hist = np.bincount(gray.ravel(), minlength=256)
    probs = hist[hist > 0] / gray.size
    entropy = float(-(probs * np.log2(probs)).sum())
    # Localized brigh object check
    # Threshold dim pixels to catch faint animals
    bright_mask = (gray > 40).astype(np.uint8)
    # Remove tiny isolated noise pixels
    kernel = np.ones((3, 3), np.uint8)
    bright_mask_clean = cv2.morphologyEx(bright_mask, cv2.MORPH_OPEN, kernel)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(bright_mask_clean,connectivity=8)
    frame_area = gray.shape[0] * gray.shape[1]

    if num_labels <= 1:
        largest_bright_component_area = 0
        num_bright_components_area_50 = 0
        num_bright_components_area_200 = 0
    else:
        component_areas = stats[1:, cv2.CC_STAT_AREA]  # skip background label 0

        largest_bright_component_area = int(component_areas.max())
        num_bright_components_area_50 = int((component_areas >= 50).sum())
        num_bright_components_area_200 = int((component_areas >= 200).sum())

    largest_bright_component_area_ratio = float(
        largest_bright_component_area / frame_area
    )

    return {
        "mean_brightness": mean_brightness,
        "median_brightness": median_brightness,
        "max_brightness": max_brightness,
        "p95_brightness": p95_brightness,
        "p99_brightness": p99_brightness,
        "p999_brightness": p999_brightness,
        "pct_pixels_above_20": pct_pixels_above_20,
        "pct_pixels_above_40": pct_pixels_above_40,
        "pct_pixels_above_60": pct_pixels_above_60,
        "pct_pixels_above_80": pct_pixels_above_80,
        "entropy": entropy,
        "largest_bright_component_area": largest_bright_component_area,
        "largest_bright_component_area_ratio": largest_bright_component_area_ratio,
        "num_bright_components_area_50": num_bright_components_area_50,
        "num_bright_components_area_200": num_bright_components_area_200,
    }

In [ ]:
def is_black_frame(metrics):
    return (
        metrics["mean_brightness"] <= BLACK_RULE["mean_brightness_max"]
        and metrics["median_brightness"] <= BLACK_RULE["median_brightness_max"]
        and metrics["p99_brightness"] <= BLACK_RULE["p99_brightness_max"]
        and metrics["p999_brightness"] <= BLACK_RULE["p999_brightness_max"]
        and metrics["pct_pixels_above_20"] <= BLACK_RULE["pct_pixels_above_20_max"]
        and metrics["pct_pixels_above_40"] <= BLACK_RULE["pct_pixels_above_40_max"]
        and metrics["pct_pixels_above_60"] <= BLACK_RULE["pct_pixels_above_60_max"]
        and metrics["entropy"] <= BLACK_RULE["entropy_max"]
        and metrics["largest_bright_component_area"] <= BLACK_RULE["largest_bright_component_area_max"]
    )

In [ ]:
def sample_frame_indices(total_frames, fps):
    """
    Randomly samples candidate frame indices from a video. The selected candidate frames are at least MIN_SECONDS_APART apart.
    """
    if total_frames <= 0:
        return []
    min_gap = int(round(fps * MIN_SECONDS_APART))
    if min_gap <= 0:
        min_gap = FPS * MIN_SECONDS_APART
    # Random offset prevents always checking frame 0, 260, 520, etc.
    max_offset = min(min_gap - 1, max(total_frames - 1, 0))
    offset = random.randint(0, max_offset)
    possible_frames = list(range(offset, total_frames, min_gap))
    if len(possible_frames) == 0:
        return []
    n_to_sample = min(MAX_FRAMES_PER_VID, len(possible_frames))
    sampled = random.sample(possible_frames, k=n_to_sample)
    return sorted(sampled)

In [103]:
def make_balanced_video_order(camera_df):
    """
    Creates a randomized video order that spreads videos across different hours to avoid sampling from same time every day.
    """
    groups = {}
    for hour, sub_df in camera_df.groupby("hour"):
        records = sub_df.to_dict("records")
        random.shuffle(records)
        groups[hour] = records
    ordered = []
    while groups:
        hours = list(groups.keys())
        random.shuffle(hours)
        for hour in hours:
            if hour not in groups:
                continue
            if len(groups[hour]) == 0:
                del groups[hour]
                continue
            ordered.append(groups[hour].pop())
            if len(groups[hour]) == 0:
                del groups[hour]
    return ordered

In [104]:
def make_camera_targets(camera_names, total_target):
    """
    Splits the total target approximately evenly across cameras.
    """
    base = total_target // len(camera_names)
    remainder = total_target % len(camera_names)
    targets = {}
    for i, camera in enumerate(camera_names):
        targets[camera] = base + (1 if i < remainder else 0)
    return targets

### Build internal video list

In [105]:
video_df = build_video_table(ROOT)
if len(video_df) == 0:
    raise RuntimeError("No videos found. Check ROOT path and filename format.")
camera_names = sorted(video_df["camera"].unique())
print("Total videos found:", len(video_df))
print("Cameras found:", camera_names)
print()
print(video_df["camera"].value_counts())

Total videos found: 8187
Cameras found: ['10.0.11.2', '10.0.12.2', '10.0.16.2']

camera
10.0.12.2    2730
10.0.11.2    2729
10.0.16.2    2728
Name: count, dtype: int64


### Extract black frames

In [ ]:
camera_targets = make_camera_targets(camera_names, TARGET_BLACK_FRAMES)
print()
print("Target black frames per camera:")
for camera, target in camera_targets.items():
    print(f"  {camera}: {target}")

black_rows = []
saved_count_by_camera = {camera: 0 for camera in camera_names}
global_black_frame_id = 1

for camera in camera_names:
    camera_df = video_df[video_df["camera"] == camera].copy()
    video_order = make_balanced_video_order(camera_df)
    target_for_camera = camera_targets[camera]
    progress = tqdm(
        total=target_for_camera,
        desc=f"Saving black frames for {camera}",
        unit="frame"
    )
    videos_checked = 0
    for row in video_order:
        if saved_count_by_camera[camera] >= target_for_camera:
            break
        videos_checked += 1
        video_path = row["video_path"]
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue
        try:
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps is None or fps <= 0:
                fps = FPS  
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            frame_indices = sample_frame_indices(total_frames=total_frames,fps=fps)

            for frame_index in frame_indices:
                if saved_count_by_camera[camera] >= target_for_camera:
                    break
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
                ok, frame = cap.read()
                if not ok or frame is None:
                    continue
                metrics = compute_frame_metrics(frame)
                if not is_black_frame(metrics):
                    continue
                if saved_count_by_camera[camera] >= target_for_camera:
                    break
                image_name = (
                    f"black_{global_black_frame_id:04d}__"
                    f"{camera}__"
                    f"{row['video'].replace('.mp4', '')}__"
                    f"frame_{frame_index:06d}.png"
                )
                image_path = BLACK_FRAME_DIR / image_name
                image_path.parent.mkdir(parents=True, exist_ok=True)
                saved_ok = cv2.imwrite(str(image_path), frame)
                if not saved_ok:
                    raise RuntimeError(f"cv2.imwrite failed for: {image_path}")
                black_rows.append({
                    "camera": camera,
                    "year": row["year"],
                    "month": row["month"],
                    "day": row["day"],
                    "hour": row["hour"],
                    "minute": row["minute"],
                    "frame_index": frame_index,
                    "image_name": image_name,
                })
                saved_count_by_camera[camera] += 1
                global_black_frame_id += 1

                progress.update(1)
                progress.set_postfix({
                    "saved": saved_count_by_camera[camera],
                    "target": target_for_camera,
                    "videos_checked": videos_checked,
                })
        finally:
            cap.release()
    progress.close()
    print(
        f"Finished {camera}: "
        f"{saved_count_by_camera[camera]} / {target_for_camera}"
    )


Target black frames per camera:
  10.0.11.2: 50
  10.0.12.2: 50
  10.0.16.2: 50


Saving black frames for 10.0.11.2: 100%|██████████| 50/50 [00:22<00:00,  2.25frame/s, saved=50, target=50, videos_checked=50]


Finished 10.0.11.2: 50 / 50


Saving black frames for 10.0.12.2: 100%|██████████| 50/50 [00:22<00:00,  2.21frame/s, saved=50, target=50, videos_checked=50]


Finished 10.0.12.2: 50 / 50


Saving black frames for 10.0.16.2: 100%|██████████| 50/50 [00:23<00:00,  2.09frame/s, saved=50, target=50, videos_checked=50]

Finished 10.0.16.2: 50 / 50


### Save final csv table

In [110]:
black_df = pd.DataFrame(black_rows)
# Keep only the first TARGET_BLACK_FRAMES just in case
black_df = black_df.head(TARGET_BLACK_FRAMES).copy()
# Re-number cleanly from 1 to N
if len(black_df) > 0:
    black_df["black_frame_id"] = range(1, len(black_df) + 1)
black_csv = OUT / "black_frame_manifest.csv"
black_df.to_csv(black_csv, index=False)

print()
print("Done.")
print("Saved black frames:", len(black_df))
print("Images saved in:", BLACK_FRAME_DIR)
print("Manifest saved to:", black_csv)
print()
print("Saved by camera:")
print(black_df["camera"].value_counts() if len(black_df) > 0 else "No frames saved.")

if len(black_df) < TARGET_BLACK_FRAMES:
    print()
    print("Warning: fewer black frames were found than requested.")
    print("Try relaxing BLACK_RULE thresholds near the top of the notebook.")
    print("For example:")
    print("  mean_brightness_max: 10 or 12")
    print("  median_brightness_max: 8 or 10")
    print("  pct_pixels_above_20_max: 0.02")
    print("  entropy_max: 4.0")
black_df.head()


Done.
Saved black frames: 150
Images saved in: /Users/alopias/Desktop/Huyen-deePi/deePi-laptop/black_frames
Manifest saved to: /Users/alopias/Desktop/Huyen-deePi/deePi-laptop/black_frame_manifest.csv

Saved by camera:
camera
10.0.11.2    50
10.0.12.2    50
10.0.16.2    50
Name: count, dtype: int64


,camera,year,month,day,hour,minute,frame_index,image_name,black_frame_id
0,10.0.11.2,2023,8,9,3,46,1706,black_0001__10.0.11.2__202308090346__frame_001...,1
1,10.0.11.2,2023,11,2,15,46,198,black_0002__10.0.11.2__202311021546__frame_000...,2
2,10.0.11.2,2023,7,1,10,48,163,black_0003__10.0.11.2__202307011048__frame_000...,3
3,10.0.11.2,2023,7,1,23,46,554,black_0004__10.0.11.2__202307012346__frame_000...,4
4,10.0.11.2,2023,9,11,11,46,1058,black_0005__10.0.11.2__202309111146__frame_001...,5
